# 02 — Fine-Tune Agent 1: Security Requirements Extraction (`qwen2.5-3b-cc-agent1`)

Fine-tunes `unsloth/Qwen2.5-3B-Instruct` (4-bit QLoRA, `r=16`, `lora_alpha=16`) on Common Criteria SFR-to-Requirements pairs so the model emits structured JSON requirements with reasoning and source citations.

In [ ]:
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# 0. Force Unsloth-compatible stable versions to resolve the Colab conflict
!pip install -q "transformers<5.0.0" "trl<=1.13.0" "datasets<4.4.0" "unsloth-zoo>=2026.9.0"

# 1. MUST BE THE VERY FIRST IMPORT to apply GPU optimizations
import unsloth
import json
from datasets import Dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import get_chat_template

# 2. Automatically Recreate Your Dataset
dataset_content = [
  {
    "functionality": "Role-Based Access Control (RBAC) enforcement in operating systems",
    "retrieved_context": [
      {"sfr_id": "FDP_ACC.2.1", "source_pp": "Korean National Protection Profile for Access Control in OS V3.0", "text": "The TSF shall enforce a Role-based access control Policy (or Mandatory access control Policy) on a defined list of subjects and objects, and operations among subjects and objects covered by the SFP. The TSF shall ensure that all operations among all subjects and objects controlled by the TSF are covered by the access control policy SFP."},
      {"sfr_id": "FMT_SMR.1.1", "source_pp": "Korean National Protection Profile for Access Control in OS V3.0", "text": "The TSF shall maintain the authorized identified roles, and shall be able to associate users with their roles."},
      {"sfr_id": "FMT_MSA.3.1", "source_pp": "Korean National Protection Profile for Access Control in OS V3.0", "text": "The TSF shall enforce the access control SFP to provide restrictive (or permissive) default values for security attributes that are used to enforce the SFP."},
      {"sfr_id": "FMT_MOF.1.1", "source_pp": "Korean National Protection Profile for Access Control in OS V3.0", "text": "The TSF shall restrict the ability to conduct management actions of the security functions to the authorized administrator."}
    ],
    "requirements": [
      {"req_id": "REQ-001", "statement": "The system shall enforce a role-based access control policy covering all subjects, objects, and operations, with no operation left outside the policy's coverage.", "category": "Security - Access Control", "source_ref": "FDP_ACC.2.1 (Korean National PP for Access Control in OS V3.0)", "reasoning": "A partial RBAC implementation that only governs some operations would leave gaps an attacker could exploit by finding an uncovered operation to bypass role restrictions entirely. FDP_ACC.2.1's 'Complete' access control requirement (all subjects/objects/operations covered) closes this gap by eliminating any possibility of an unmediated, policy-free path through the system."},
      {"req_id": "REQ-002", "statement": "The system shall maintain a defined set of authorized roles and shall associate individual users with those roles.", "category": "Security - Management", "source_ref": "FMT_SMR.1.1 (Korean National PP for Access Control in OS V3.0)", "reasoning": "REQ-001 requires enforcement of a role-based policy, but that's meaningless without roles actually existing as first-class, distinctly maintained entities that users are genuinely mapped to - otherwise 'role-based' access control could collapse into a single undifferentiated permission set. FMT_SMR.1.1 closes this gap by requiring genuine role maintenance and user-role association as separate, auditable state."},
      {"req_id": "REQ-003", "statement": "The system shall provide restrictive default values for security attributes used to enforce the access control policy, unless explicitly configured otherwise.", "category": "Security - Access Control", "source_ref": "FMT_MSA.3.1 (Korean National PP for Access Control in OS V3.0)", "reasoning": "Without this requirement, a newly created object or user could default to overly permissive access rights before an administrator explicitly restricts them, creating a window of excessive exposure. FMT_MSA.3.1 closes this gap by requiring restrictive defaults, so the secure state is the starting point rather than something that has to be actively configured after the fact."},
      {"req_id": "REQ-004", "statement": "The system shall restrict the ability to perform security management actions to the authorized administrator role only.", "category": "Security - Management", "source_ref": "FMT_MOF.1.1 (Korean National PP for Access Control in OS V3.0)", "reasoning": "REQ-002 and REQ-003 establish roles and secure defaults, but if any user (not just administrators) could alter role definitions or security-relevant defaults, the entire RBAC structure could be undermined from within. FMT_MOF.1.1 closes this gap by explicitly restricting management of the security functions themselves to the administrator role, protecting the RBAC configuration from being modified by the very users it's meant to constrain."}
    ]
  },
  {
    "functionality": "Firewall traffic filtering / access control policy enforcement",
    "retrieved_context": [
      {"sfr_id": "FDP_IFF.1.1", "source_pp": "Firewall with high requirements, Version 2.2", "text": "The TSF shall enforce the information flow control SFP based on: network origin/destination identity, user origin/destination identity, type of application (FTP, SQL, HTTP, SMTP, TELNET), type of application command requested, date/time of access, number/frequency/throughput of communication flow."},
      {"sfr_id": "FDP_ACC.2", "source_pp": "Firewall with high requirements, Version 2.2", "text": "Complete access control - the TSF shall enforce the access control policy on all subjects and objects, and all operations among subjects and objects covered by the policy."},
      {"sfr_id": "FAU_GEN.1", "source_pp": "Firewall with high requirements, Version 2.2", "text": "The TSF shall generate an audit record of the following auditable events: start-up and shutdown of audit functions, and all auditable events for the not-specified level of audit."},
      {"sfr_id": "FMT_MSA.1.1", "source_pp": "Firewall with high requirements, Version 2.2", "text": "The TSF shall enforce the access control SFP / information flow control SFP to restrict the ability to change_default, query, modify, delete the security attributes to the authorised identified roles."},
      {"sfr_id": "FPT_RVM.1.1", "source_pp": "Firewall with high requirements, Version 2.2", "text": "The TSF shall ensure that TSP enforcement functions are invoked and succeed before each function within the TSC is allowed to proceed."}
    ],
    "requirements": [
      {"req_id": "REQ-001", "statement": "The system shall enforce an information flow control policy that filters network traffic based on network origin/destination identity, user origin/destination identity, application type, application-layer commands, date/time of access, and the number/frequency/throughput of the communication flow.", "category": "Security - Access Control", "source_ref": "FDP_IFF.1.1 (Firewall with high requirements, v2.2)", "reasoning": "Without enforcement based on the full set of policy-relevant traffic attributes, an attacker could bypass a coarse filtering policy by using traffic that matches permitted network/user identities or application types while varying other conditions such as access time, application commands, or communication rate. FDP_IFF.1.1 requires the information-flow policy to be enforced using all of these attributes together, so the requirement ensures filtering decisions cannot ignore any single policy-relevant condition."},
      {"req_id": "REQ-002", "statement": "The system shall apply the access control policy to all subjects, objects, and operations covered by the policy, with no unmediated bypass paths.", "category": "Security - Access Control", "source_ref": "FDP_ACC.2 (Firewall with high requirements, v2.2)", "reasoning": "FDP_ACC.2 (Complete access control) is stricter than FDP_ACC.1 (Subset access control) - it requires the policy to cover every subject/object/operation, not just a defined subset. This directly implies the firewall's rule engine cannot have implicit 'default allow' gaps; every traffic flow must be evaluated against the policy."},
      {"req_id": "REQ-003", "statement": "The system shall generate audit records for start-up and shutdown of the audit functions, and for all auditable events at the not-specified audit level.", "category": "Security - Audit", "source_ref": "FAU_GEN.1 (Firewall with high requirements, v2.2)", "reasoning": "Without records of audit-function startup and shutdown, an interruption or restart of auditing could create gaps in the security audit trail and make it difficult to determine when auditing was unavailable. FAU_GEN.1 requires these audit-function lifecycle events, along with the auditable events covered by the specified audit level, to be recorded, providing the evidence needed to reconstruct relevant security activity during that window."},
      {"req_id": "REQ-004", "statement": "The system shall restrict the ability to change, query, modify, or delete security attributes (filtering rules, policy configuration) to authorised administrative roles only.", "category": "Security - Management", "source_ref": "FMT_MSA.1.1 (Firewall with high requirements, v2.2)", "reasoning": "Even a perfectly designed filtering policy (REQ-001, REQ-002) is worthless if an unauthorised user can silently modify the rule set. FMT_MSA.1.1 closes this gap by tying every configuration-changing operation to role-based authorisation, preventing privilege escalation or insider tampering with the firewall's own policy."},
      {"req_id": "REQ-005", "statement": "The system shall ensure that its policy enforcement functions are invoked and succeed before any traffic-affecting function is allowed to proceed, with no path that bypasses enforcement.", "category": "Security - Access Control", "source_ref": "FPT_RVM.1.1 (Firewall with high requirements, v2.2)", "reasoning": "REQ-002 (Complete access control) defines the scope of what must be checked, but doesn't by itself guarantee the check always executes first. FPT_RVM.1.1 closes the remaining gap: a race condition or initialization-order bug could let traffic through before the filter engine is ready. This requirement forces enforcement-before-action as an architectural invariant, not just a policy scope."}
    ]
  },
  {
    "functionality": "Intrusion Prevention System (IPS) traffic analysis and threat response",
    "retrieved_context": [
      {"sfr_id": "IPS_SBD_EXT.1.1", "source_pp": "PP-Module for Intrusion Prevention Systems (IPS), Version 1.0", "text": "The TSF shall support inspection of packet header contents and be able to inspect at least the following header fields: IPv4 (version, header length, packet length, ID, IP flags, fragment offset, TTL, protocol, header checksum, source/destination address, IP options); IPv6 (version, payload length, next header, hop limit, source/destination address, routing header)."},
      {"sfr_id": "IPS_ABD_EXT.1.1", "source_pp": "PP-Module for Intrusion Prevention Systems (IPS), Version 1.0", "text": "The TSF shall support the definition of baselines ('expected and approved') and anomaly ('unexpected') traffic patterns including the specification of throughput, time of day, frequency, and thresholds."},
      {"sfr_id": "IPS_NTA_EXT.1.1", "source_pp": "PP-Module for Intrusion Prevention Systems (IPS), Version 1.0", "text": "The TSF shall perform analysis of IP-based network traffic forwarded to the TOE's sensor interfaces, and detect violations of administratively-defined IPS policies."},
      {"sfr_id": "IPS_EXT.1.4", "source_pp": "PP-Module for Intrusion Prevention Systems (IPS), Version 1.0", "text": "In inline mode: block/drop the traffic flow, with options to allow all remaining traffic flow, allow with defined exceptions, or modify and forward packets before they pass through the TOE."},
      {"sfr_id": "FAU_STG.1.1", "source_pp": "PP-Module for Intrusion Prevention Systems (IPS), Version 1.0", "text": "The TSF shall protect the stored audit records (IPS data) from unauthorized deletion, and shall be able to prevent unauthorized modifications to the stored audit records in the audit trail."}
    ],
    "requirements": [
      {"req_id": "REQ-001", "statement": "The system shall inspect IPv4 and IPv6 packet header fields (including source/destination address, protocol, TTL/hop limit, and IP options/routing header) as the basis for signature-based detection.", "category": "Security - Detection", "source_ref": "IPS_SBD_EXT.1.1 (PP-Module for IPS, v1.0)", "reasoning": "A signature-based IPS that only checks source/destination IP would miss attacks that manipulate other header fields (e.g. fragment offset abuse for evasion, TTL manipulation for reconnaissance). IPS_SBD_EXT.1.1 closes this gap by mandating inspection of the full defined field set, ensuring known-attack signatures can be matched against the complete packet context rather than a partial view."},
      {"req_id": "REQ-002", "statement": "The system shall support defining traffic baselines and anomaly patterns based on throughput, time of day, frequency, and configurable thresholds.", "category": "Security - Detection", "source_ref": "IPS_ABD_EXT.1.1 (PP-Module for IPS, v1.0)", "reasoning": "Signature-based detection (REQ-001) only catches known attack patterns; a zero-day or novel attack technique with no matching signature would pass through undetected. IPS_ABD_EXT.1.1 closes this gap by requiring anomaly-based detection - flagging traffic that deviates from an established behavioral baseline even when no known signature matches."},
      {"req_id": "REQ-003", "statement": "The system shall analyze IP-based network traffic on its sensor interfaces and detect violations of administratively-defined IPS policies.", "category": "Security - Detection", "source_ref": "IPS_NTA_EXT.1.1 (PP-Module for IPS, v1.0)", "reasoning": "Without this requirement, detection logic (REQ-001, REQ-002) could exist without ever being connected to actual live traffic - i.e. the engine could be correctly built but never actually applied. IPS_NTA_EXT.1.1 closes this gap by mandating that traffic actually forwarded to sensor interfaces is analyzed against policy in practice, not just capable of being analyzed in theory."},
      {"req_id": "REQ-004", "statement": "When operating in inline mode, the system shall be able to block or drop malicious traffic flows in real time, with configurable handling of remaining traffic.", "category": "Security - Response", "source_ref": "IPS_EXT.1.4 (PP-Module for IPS, v1.0)", "reasoning": "Detecting a threat (REQ-001/002/003) without the ability to act on it in real time would make the system a passive IDS, not an IPS - the attack would already have reached its target by the time an alert is reviewed. IPS_EXT.1.4 closes this gap by requiring inline blocking capability, converting detection into active prevention."},
      {"req_id": "REQ-005", "statement": "The system shall protect stored IPS audit records from unauthorized deletion and modification.", "category": "Security - Audit", "source_ref": "FAU_STG.1.1 (PP-Module for IPS, v1.0)", "reasoning": "An attacker who successfully evades detection (or who compromises the IPS itself) could otherwise erase or alter the audit trail to hide evidence of the intrusion attempt, defeating any post-incident investigation. FAU_STG.1.1 closes this gap by making the audit trail itself tamper-resistant, ensuring forensic evidence of detected/blocked events survives even an attempted cover-up."}
    ]
  },
  {
    "functionality": "VPN gateway network access control and user authentication",
    "retrieved_context": [
      {"sfr_id": "FDP_ACC.1", "source_pp": "Low Assurance Protection Profile for a VPN gateway, Version 1.4", "text": "The TSF shall enforce the SFP_VPN access control policy on the defined subjects, objects, and operations covered by that policy."},
      {"sfr_id": "FDP_ACF.1.2", "source_pp": "Low Assurance Protection Profile for a VPN gateway, Version 1.4", "text": "The TSF shall enforce the following rules to determine if an operation among controlled subjects and controlled objects is allowed: S.USER is only allowed to R.CONNECT to O.NETWORK if it is identified and authenticated on behalf of its security attribute."},
      {"sfr_id": "FIA_UID.2.1", "source_pp": "Low Assurance Protection Profile for a VPN gateway, Version 1.4", "text": "The TSF shall require each user to identify itself before allowing any other TSF-mediated actions on behalf of that user."},
      {"sfr_id": "FIA_UAU.2.1", "source_pp": "Low Assurance Protection Profile for a VPN gateway, Version 1.4", "text": "The TSF shall require each user to be successfully authenticated before allowing any other TSF-mediated actions on behalf of that user."}
    ],
    "requirements": [
      {"req_id": "REQ-001", "statement": "The system shall enforce the VPN access control policy on all defined subjects, objects, and operations covered by that policy.", "category": "Security - Access Control", "source_ref": "FDP_ACC.1 (Low Assurance PP for VPN gateway, v1.4)", "reasoning": "Without a defined scope of enforcement, a VPN gateway could have ad-hoc or inconsistent access decisions for different connection types. FDP_ACC.1 closes this gap by requiring a single, defined policy (SFP_VPN) to govern every covered subject/object/operation, giving the gateway one consistent basis for connection decisions rather than scattered per-case logic."},
      {"req_id": "REQ-002", "statement": "The system shall only permit a user to connect to the protected network (O.NETWORK) if that user has been identified and authenticated according to their assigned security attributes.", "category": "Security - Access Control", "source_ref": "FDP_ACF.1.2 (Low Assurance PP for VPN gateway, v1.4)", "reasoning": "Even with a defined access control policy in scope (REQ-001), the policy is meaningless unless it's tied to a concrete condition. FDP_ACF.1.2 closes this gap by binding network connection specifically to identity/authentication status - an unauthenticated party attempting to connect through the VPN gateway has no path to R.CONNECT regardless of other policy state."},
      {"req_id": "REQ-003", "statement": "The system shall require each user to identify itself before any other TSF-mediated action is permitted on that user's behalf.", "category": "Security - Identification", "source_ref": "FIA_UID.2.1 (Low Assurance PP for VPN gateway, v1.4)", "reasoning": "REQ-002 assumes identification already happened, but without this requirement there's no guarantee identification is mandatory before any action - a system could allow default/anonymous actions before identity is established. FIA_UID.2.1 closes this gap by making identification a strict precondition for any TSF-mediated action, not just for network connection specifically."},
      {"req_id": "REQ-004", "statement": "The system shall require each user to be successfully authenticated before any other TSF-mediated action is permitted on that user's behalf.", "category": "Security - Authentication", "source_ref": "FIA_UAU.2.1 (Low Assurance PP for VPN gateway, v1.4)", "reasoning": "Identification alone (REQ-003) only establishes a claimed identity - it doesn't verify it. A user could claim any identity without this requirement. FIA_UAU.2.1 closes this gap by mandating successful authentication (proof of identity) as a precondition, preventing identity spoofing from granting network access through the gateway."}
    ]
  },
  {
    "functionality": "Network gateway secure configuration and role-based management",
    "retrieved_context": [
      {"sfr_id": "FMT_MOF.1.1", "source_pp": "Configurable Security Guard (CSG), Version 3.3", "text": "The TSF shall restrict the ability to determine the behaviour of, disable, enable, modify the behaviour of the functions to the authorised identified roles."},
      {"sfr_id": "FMT_SMR.2.1", "source_pp": "Configurable Security Guard (CSG), Version 3.3", "text": "The TSF shall maintain the roles: security officer, TOE operator and TOE administrator. The TSF shall be able to associate users with roles, and shall ensure that the conditions for the different roles are satisfied."},
      {"sfr_id": "FMT_REV.1.1", "source_pp": "Configurable Security Guard (CSG), Version 3.3", "text": "The TSF shall restrict the ability to revoke security attributes associated with users or objects within the TSC to the security officer, TOE operator and TOE administrator, and shall enforce defined revocation rules."},
      {"sfr_id": "FPT_RPL.1.1", "source_pp": "Configurable Security Guard (CSG), Version 3.3", "text": "The TSF shall detect replay for user authentication messages, and shall perform specific actions when replay is detected."},
      {"sfr_id": "FPT_SEP.1.1", "source_pp": "Configurable Security Guard (CSG), Version 3.3", "text": "The TSF shall maintain a security domain for its own execution that protects it from interference and tampering by untrusted subjects, and shall enforce separation between the security domains of subjects in the TSC."}
    ],
    "requirements": [
      {"req_id": "REQ-001", "statement": "The system shall restrict the ability to determine, enable, disable, or modify the behaviour of security-relevant gateway functions to authorised identified roles.", "category": "Security - Management", "source_ref": "FMT_MOF.1.1 (Configurable Security Guard v3.3)", "reasoning": "Without this requirement, any user reaching the management interface could reconfigure core gateway behavior (e.g. disable filtering), regardless of their actual authorization level. FMT_MOF.1.1 closes this gap by tying every behavior-altering management operation to explicit role authorisation, not just general authentication."},
      {"req_id": "REQ-002", "statement": "The system shall maintain distinct security officer, operator, and administrator roles, associate users with those roles, and enforce the conditions required for each role.", "category": "Security - Management", "source_ref": "FMT_SMR.2.1 (Configurable Security Guard v3.3)", "reasoning": "REQ-001 assumes 'authorised roles' already exist and are meaningfully distinct, but without this requirement a system could implement only a single flat admin role, collapsing the separation-of-duties principle. FMT_SMR.2.1 closes this gap by mandating genuinely distinct roles with enforced membership conditions, so REQ-001's role-based restriction has real substance behind it."},
      {"req_id": "REQ-003", "statement": "The system shall restrict the ability to revoke security attributes of users or objects to the security officer, operator, and administrator roles, and shall enforce defined revocation rules.", "category": "Security - Management", "source_ref": "FMT_REV.1.1 (Configurable Security Guard v3.3)", "reasoning": "Role assignment (REQ-002) alone doesn't address what happens when a user's access needs to be withdrawn - e.g. after a compromise or role change. FMT_REV.1.1 closes this gap by requiring a controlled, role-restricted revocation mechanism, preventing a scenario where a former or compromised user's attributes remain valid indefinitely."},
      {"req_id": "REQ-004", "statement": "The system shall detect replay of user authentication messages and perform a defined response action when replay is detected.", "category": "Security - Authentication", "source_ref": "FPT_RPL.1.1 (Configurable Security Guard v3.3)", "reasoning": "Role-based access control (REQ-001 to REQ-003) assumes each authentication event reflects a genuine, fresh login - but a captured authentication message could be replayed by an attacker to impersonate a legitimate role-holder without knowing their credentials. FPT_RPL.1.1 closes this gap by requiring active replay detection, protecting the integrity of the role-based system those other requirements depend on."},
      {"req_id": "REQ-005", "statement": "The system shall maintain a protected security domain for its own execution, isolated from interference or tampering by untrusted subjects, and shall enforce separation between subject domains.", "category": "Security - Self-Protection", "source_ref": "FPT_SEP.1.1 (Configurable Security Guard v3.3)", "reasoning": "All the access-control and management requirements above (REQ-001 to REQ-004) assume the TSF itself is trustworthy and cannot be tampered with by the very subjects it's supposed to be controlling. FPT_SEP.1.1 closes this foundational gap by requiring domain separation - without it, an untrusted process could potentially modify the security-enforcing code itself, invalidating every other requirement in this set."}
    ]
  },
  {
    "functionality": "Database data-at-rest encryption",
    "retrieved_context": [
      {"sfr_id": "FCS_CKM.1.1", "source_pp": "Korean National Protection Profile for Database Encryption V1.0", "text": "The TSF shall generate cryptographic keys in accordance with a specified cryptographic key generation algorithm and specified cryptographic key sizes that meet defined standards."},
      {"sfr_id": "FDP_UDE.1.1", "source_pp": "Korean National Protection Profile for Database Encryption V1.0", "text": "The TSF shall provide TOE users with the ability to encrypt/decrypt user data according to the list of encryption/decryption methods specified."},
      {"sfr_id": "FCS_COP.1.1", "source_pp": "Korean National Protection Profile for Database Encryption V1.0", "text": "The TSF shall perform the list of cryptographic operations in accordance with a specified cryptographic algorithm and cryptographic key sizes that meet defined standards. This SFR is the operation required by FDP_UDE.1 User Data Encryption."},
      {"sfr_id": "FCS_CKM.4.1", "source_pp": "Korean National Protection Profile for Database Encryption V1.0", "text": "The TSF shall destruct cryptographic keys in accordance with a specified cryptographic key destruction method that meets defined standards. This SFR applies to all cryptographic keys covered in FCS_CKM.1."},
      {"sfr_id": "FDP_RIP.1.1", "source_pp": "Korean National Protection Profile for Database Encryption V1.0", "text": "The TSF shall ensure that any previous information content of a resource is made unavailable upon the allocation or deallocation of the resource, for user data. 'Not available' means unrecoverable deletion."}
    ],
    "requirements": [
      {"req_id": "REQ-001", "statement": "The system shall generate cryptographic keys using a specified key generation algorithm and key sizes that meet defined standards.", "category": "Security - Cryptography", "source_ref": "FCS_CKM.1.1 (Korean National PP for Database Encryption v1.0)", "reasoning": "A database encryption scheme is only as strong as its weakest key - a weak or predictable key generation method would let an attacker derive keys without breaking the encryption algorithm itself. FCS_CKM.1.1 closes this gap by mandating standards-compliant key generation, ensuring the cryptographic strength claimed by the encryption operation actually holds in practice."},
      {"req_id": "REQ-002", "statement": "The system shall provide the ability for authorized users to encrypt and decrypt user data using a specified set of encryption/decryption methods.", "category": "Security - Data Protection", "source_ref": "FDP_UDE.1.1 (Korean National PP for Database Encryption v1.0)", "reasoning": "Key generation alone (REQ-001) doesn't guarantee that data is actually encrypted - a system could generate valid keys while still storing data in plaintext. FDP_UDE.1.1 closes this gap by making encryption/decryption a functional capability the system must actually expose and apply to user data, not just a supporting cryptographic primitive."},
      {"req_id": "REQ-003", "statement": "The system shall perform cryptographic operations in accordance with a specified algorithm and key sizes meeting defined standards, as required to support user data encryption.", "category": "Security - Cryptography", "source_ref": "FCS_COP.1.1 (Korean National PP for Database Encryption v1.0)", "reasoning": "REQ-002 requires an encryption capability, but without this requirement the actual cryptographic algorithm used to implement it could be arbitrary or non-standard (e.g. a weak or proprietary cipher). FCS_COP.1.1 closes this gap by mandating that the specific operation backing FDP_UDE.1 conforms to a defined, standards-based algorithm - preventing 'encryption' in name only."},
      {"req_id": "REQ-004", "statement": "The system shall destroy cryptographic keys using a specified key destruction method that meets defined standards, covering all keys generated under this policy.", "category": "Security - Cryptography", "source_ref": "FCS_CKM.4.1 (Korean National PP for Database Encryption v1.0)", "reasoning": "A key that is merely deleted via a normal file-delete operation may remain recoverable in memory or storage remnants, letting an attacker who gains later access recover old keys and decrypt historical data. FCS_CKM.4.1 closes this gap by requiring a defined, standards-compliant destruction method that ensures keys are genuinely unrecoverable once retired."},
      {"req_id": "REQ-005", "statement": "The system shall ensure that any previous content of a resource is made unrecoverable upon allocation or deallocation of that resource, for user data.", "category": "Security - Data Protection", "source_ref": "FDP_RIP.1.1 (Korean National PP for Database Encryption v1.0)", "reasoning": "Even with encryption (REQ-002/003) and proper key destruction (REQ-004), decrypted plaintext data temporarily held in memory or storage buffers during processing could remain recoverable after the resource is freed and reallocated to another process. FDP_RIP.1.1 closes this residual-data gap by mandating that leftover content is made genuinely unrecoverable, not just logically 'deleted'."}
    ]
  },
  {
    "functionality": "TLS-based data-in-transit protection",
    "retrieved_context": [
      {"sfr_id": "FCS_TLSC_EXT.1.1", "source_pp": "Functional Package for Transport Layer Security (TLS) Version 2.0", "text": "The TSF shall implement TLS 1.2 (RFC 5246) and (optionally) TLS 1.3 (RFC 8446) as a client that supports additional functionality for session renegotiation protection, and shall abort attempts by a server to negotiate all other TLS or SSL versions."},
      {"sfr_id": "FCS_TLSC_EXT.1.3", "source_pp": "Functional Package for Transport Layer Security (TLS) Version 2.0", "text": "The TSF shall not offer ciphersuites indicating: the null encryption component; support for anonymous servers; use of deprecated or export-grade cryptography including DES, 3DES, RC2, RC4, or IDEA; use of MD5 - and shall abort sessions where a server attempts to negotiate ciphersuites not enumerated in the client hello message."},
      {"sfr_id": "FCS_TLSC_EXT.2.1", "source_pp": "Functional Package for Transport Layer Security (TLS) Version 2.0", "text": "The TSF shall support mutual authentication using X.509v3 certificates during the handshake, including support for post-handshake authentication."},
      {"sfr_id": "FCS_TLSC_EXT.4.1", "source_pp": "Functional Package for Transport Layer Security (TLS) Version 2.0", "text": "The TSF shall support secure renegotiation through use of the 'renegotiation_info' TLS extension in accordance with RFC 5746, and shall terminate the session if an unexpected server hello is received."},
      {"sfr_id": "FAU_GEN.1", "source_pp": "Functional Package for Transport Layer Security (TLS) Version 2.0", "text": "The TSF shall generate an audit record of the following auditable events: start-up and shutdown of the audit functions, and all auditable events for the not-specified level of audit."}
    ],
    "requirements": [
      {"req_id": "REQ-001", "statement": "The system shall implement TLS 1.2 and/or TLS 1.3 as a client, and shall abort any attempt by a server to negotiate a different (older or non-TLS) protocol version.", "category": "Security - Cryptography", "source_ref": "FCS_TLSC_EXT.1.1 (Functional Package for TLS v2.0)", "reasoning": "If a client silently accepted an older protocol (e.g. SSLv3, TLS 1.0), an attacker could perform a version-downgrade attack, forcing the connection onto a version with known, exploitable weaknesses. FCS_TLSC_EXT.1.1 closes this gap by requiring the client to actively abort the connection rather than fall back when a server proposes an unapproved version."},
      {"req_id": "REQ-002", "statement": "The system shall not offer ciphersuites using null encryption, anonymous servers, deprecated/export-grade cryptography (DES, 3DES, RC2, RC4, IDEA), or MD5, and shall abort sessions where the server negotiates a ciphersuite outside the offered list.", "category": "Security - Cryptography", "source_ref": "FCS_TLSC_EXT.1.3 (Functional Package for TLS v2.0)", "reasoning": "REQ-001 ensures the protocol version is safe, but the encryption strength still depends entirely on which ciphersuite is actually negotiated within that version - a technically valid TLS 1.2 session could still use a broken cipher like RC4. FCS_TLSC_EXT.1.3 closes this gap by explicitly excluding known-weak ciphersuites from ever being offered, and by rejecting any server attempt to substitute an unoffered one."},
      {"req_id": "REQ-003", "statement": "The system shall support mutual authentication using X.509v3 certificates during the TLS handshake, including post-handshake authentication.", "category": "Security - Authentication", "source_ref": "FCS_TLSC_EXT.2.1 (Functional Package for TLS v2.0)", "reasoning": "REQ-001/002 secure the channel's cryptography, but standard TLS by default only authenticates the server to the client, not the reverse - meaning the server has no cryptographic assurance of the client's identity. FCS_TLSC_EXT.2.1 closes this gap by requiring certificate-based mutual authentication, preventing an unauthorized client from establishing a trusted session even over an otherwise correctly-secured channel."},
      {"req_id": "REQ-004", "statement": "The system shall support secure renegotiation via the 'renegotiation_info' TLS extension per RFC 5746, and shall terminate the session if an unexpected server hello is received.", "category": "Security - Cryptography", "source_ref": "FCS_TLSC_EXT.4.1 (Functional Package for TLS v2.0)", "reasoning": "Without this requirement, session renegotiation (used for re-keying long-lived connections) is vulnerable to a known man-in-the-middle attack where an attacker injects data before a renegotiation, which the peer then attributes to the authenticated session. FCS_TLSC_EXT.4.1 closes this gap by cryptographically binding renegotiations to the original session via RFC 5746, and rejecting unexpected renegotiation attempts outright."},
      {"req_id": "REQ-005", "statement": "The system shall generate audit records for start-up and shutdown of the audit function and for all auditable events at the not-specified audit level.", "category": "Security - Audit", "source_ref": "FAU_GEN.1 (Functional Package for TLS v2.0)", "reasoning": "Even with a cryptographically sound TLS implementation (REQ-001 to REQ-004), repeated abort events (e.g. downgrade attempts, unexpected renegotiations) would go unnoticed without logging, denying administrators visibility into active attack attempts against the connection. FAU_GEN.1 closes this gap by ensuring the audit trail captures these lifecycle and security-relevant events for later review."}
    ]
  },
  {
    "functionality": "TPM-based platform integrity measurement and attestation",
    "retrieved_context": [
      {"sfr_id": "FPT_PHP.3.1", "source_pp": "Protection Profile PC Client Specific TPM Family 2.0, Level 0, Rev 1.59", "text": "The TSF shall resist physical manipulation and physical probing to the TSF by responding automatically such that the SFRs are always enforced."},
      {"sfr_id": "FPT_TST.1.1", "source_pp": "Protection Profile PC Client Specific TPM Family 2.0, Level 0, Rev 1.59", "text": "The TSF shall run a suite of self tests at the request of the authorised user (TPM2_SelfTest / TPM2_IncrementalSelfTest commands), at initialisation after reset and before the reception of the first command, and prior to execution of a command using a not-self-tested function."},
      {"sfr_id": "FCS_RNG.1.1", "source_pp": "Protection Profile PC Client Specific TPM Family 2.0, Level 0, Rev 1.59", "text": "The TSF shall provide a physical, deterministic, or hybrid random number generator that implements a defined list of security capabilities, and shall provide random numbers that meet a defined quality metric."},
      {"sfr_id": "FCO_NRO.1.1", "source_pp": "Protection Profile PC Client Specific TPM Family 2.0, Level 0, Rev 1.59", "text": "The TSF shall be able to generate evidence of origin for transmitted attestation structures (TPM2B_ATTEST) and object creation tickets at the request of the originator."}
    ],
    "requirements": [
      {"req_id": "REQ-001", "statement": "The system shall resist physical manipulation and probing of the TPM's security-enforcing hardware, responding automatically to ensure SFRs remain enforced even under physical attack.", "category": "Security - Physical Protection", "source_ref": "FPT_PHP.3.1 (PC Client Specific TPM PP, Rev 1.59)", "reasoning": "A root-of-trust component is meaningless if an attacker with physical access can simply probe it to extract keys or bypass its logic - software-level integrity guarantees are irrelevant if the hardware itself can be tampered with. FPT_PHP.3.1 closes this gap by requiring automatic, active resistance to physical manipulation, not just passive tamper-evidence."},
      {"req_id": "REQ-002", "statement": "The system shall run a defined suite of self-tests at initialisation (before processing any command), on user request, and before executing any function that has not yet been self-tested in the current session.", "category": "Security - Integrity", "source_ref": "FPT_TST.1.1 (PC Client Specific TPM PP, Rev 1.59)", "reasoning": "Even tamper-resistant hardware (REQ-001) could still fail due to a manufacturing defect, environmental fault, or firmware corruption that physical protection alone doesn't detect. FPT_TST.1.1 closes this gap by requiring active self-verification at the specific moments where undetected failure would be most dangerous - at boot, before first use, and before invoking any not-yet-verified function."},
      {"req_id": "REQ-003", "statement": "The system shall provide a physical, deterministic, or hybrid random number generator meeting a defined quality metric, used to support key generation and other cryptographic operations.", "category": "Security - Cryptography", "source_ref": "FCS_RNG.1.1 (PC Client Specific TPM PP, Rev 1.59)", "reasoning": "A TPM's entire trust model rests on the unpredictability of its generated keys and nonces - if the underlying randomness is weak or predictable, an attacker could reconstruct keys without needing to physically or logically attack the device at all. FCS_RNG.1.1 closes this gap by mandating a quality-metric-verified RNG, ensuring the foundational entropy backing every other security operation is genuinely unpredictable."},
      {"req_id": "REQ-004", "statement": "The system shall generate verifiable evidence of origin for transmitted attestation structures (TPM2B_ATTEST) and object creation tickets, at the request of the originator.", "category": "Security - Attestation", "source_ref": "FCO_NRO.1.1 (PC Client Specific TPM PP, Rev 1.59)", "reasoning": "Self-tests and randomness (REQ-002/003) establish that the TPM itself is trustworthy internally, but a remote party (e.g. a server checking platform integrity) has no way to verify that an attestation report genuinely came from this specific TPM rather than being forged or replayed by a compromised host. FCO_NRO.1.1 closes this gap by requiring non-repudiable, verifiable evidence of origin on every attestation structure, which is the actual mechanism that lets an external verifier trust the platform's integrity claims."}
    ]
  },
  {
    "functionality": "Hardware root-of-trust secure boot verification",
    "retrieved_context": [
      {"sfr_id": "FPT_INI.1.1", "source_pp": "MCU Root of Trust Protection Profile (GPT_SPE_146, v1.0)", "text": "The TOE shall provide an initialisation function which is self-protected for integrity and authenticity."},
      {"sfr_id": "FPT_INI.1.2", "source_pp": "MCU Root of Trust Protection Profile (GPT_SPE_146, v1.0)", "text": "The TOE initialisation function shall ensure that certain properties (such as authenticity, integrity, correct version) hold on certain elements (TSF/user firmware, software, or data) immediately before establishing the TSF in a secure initial state."},
      {"sfr_id": "FPT_TEE.1.1", "source_pp": "MCU Root of Trust Protection Profile (GPT_SPE_146, v1.0)", "text": "The TSF shall run a suite of tests prior to execution to check the fulfilment of authenticity of ARoT (Application Root of Trust) code."},
      {"sfr_id": "FPT_TEE.1.2", "source_pp": "MCU Root of Trust Protection Profile (GPT_SPE_146, v1.0)", "text": "If the test fails, the TSF shall not start the execution of the ARoT instance."},
      {"sfr_id": "FAU_SAR.1.1", "source_pp": "MCU Root of Trust Protection Profile (GPT_SPE_146, v1.0)", "text": "The TSF shall provide all users with the capability to read the MCU RoT identifier from the audit records."}
    ],
    "requirements": [
      {"req_id": "REQ-001", "statement": "The system shall provide an initialisation function that is itself self-protected for integrity and authenticity before any other function executes.", "category": "Security - Boot Integrity", "source_ref": "FPT_INI.1.1 (MCU Root of Trust PP, v1.0)", "reasoning": "A root of trust that verifies everything else but whose own initialisation code can be tampered with provides no real security guarantee - the verification logic itself becomes the weakest link. FPT_INI.1.1 closes this gap by requiring the initialisation function to be self-protected, meaning the very first code to execute cannot itself be silently modified."},
      {"req_id": "REQ-002", "statement": "The initialisation function shall verify defined properties (authenticity, integrity, correct version) of specified firmware, software, and data elements before establishing the TSF in a secure initial state.", "category": "Security - Boot Integrity", "source_ref": "FPT_INI.1.2 (MCU Root of Trust PP, v1.0)", "reasoning": "REQ-001 protects the initialisation code itself, but doesn't specify what that code must actually check before trusting the rest of the system. FPT_INI.1.2 closes this gap by mandating concrete verification of downstream firmware/software properties, which is the actual chain-of-trust mechanism - each stage verifies the next before handing off control."},
      {"req_id": "REQ-003", "statement": "The system shall run a test suite to verify the authenticity of Application Root of Trust (ARoT) code prior to its execution.", "category": "Security - Boot Integrity", "source_ref": "FPT_TEE.1.1 (MCU Root of Trust PP, v1.0)", "reasoning": "REQ-002 covers the core boot chain, but application-level trusted code (ARoT) loaded after the core boot process represents a separate trust boundary that the core initialisation check doesn't necessarily cover. FPT_TEE.1.1 closes this gap by requiring a distinct authenticity check specifically at the point application-level trusted code is about to run."},
      {"req_id": "REQ-004", "statement": "If the authenticity test for ARoT code fails, the system shall not start execution of that ARoT instance.", "category": "Security - Boot Integrity", "source_ref": "FPT_TEE.1.2 (MCU Root of Trust PP, v1.0)", "reasoning": "REQ-003 only requires that a test runs - without this requirement, a system could run the test, detect failure, and still proceed to execute the unverified code anyway (a 'fail-open' design). FPT_TEE.1.2 closes this gap by making failure enforcement mandatory: a failed test must actually block execution, not just log a warning."},
      {"req_id": "REQ-005", "statement": "The system shall allow authorized users to read the MCU root-of-trust identifier from the audit records.", "category": "Security - Audit", "source_ref": "FAU_SAR.1.1 (MCU Root of Trust PP, v1.0)", "reasoning": "Even with correct boot verification (REQ-001 to REQ-004), an administrator or remote verifier has no way to confirm which specific hardware root of trust instance attested to a given boot event without a readable identifier. FAU_SAR.1.1 closes this gap by requiring the RoT's identity to be retrievable from audit records, enabling traceability of which physical device's root of trust is vouching for a given system state."}
    ]
  },
  {
    "functionality": "Single Sign-On (SSO) authentication",
    "retrieved_context": [
      {"sfr_id": "FIA_UAU.5.1", "source_pp": "Korean National Protection Profile for Single Sign On V3.0", "text": "The TSF shall provide password authentication mechanism and any additional authentication mechanisms to support user authentication, and shall authenticate any user's claimed identity according to defined rules describing how the multiple authentication mechanisms provide authentication."},
      {"sfr_id": "FIA_IMA.1.1", "source_pp": "Korean National Protection Profile for Single Sign On V3.0", "text": "The TSF shall perform mutual authentication between different parts of the TOE using a defined authentication protocol that meets a defined list of standards."},
      {"sfr_id": "FTA_SSL.3.1", "source_pp": "Korean National Protection Profile for Single Sign On V3.0", "text": "The TSF shall terminate an interactive session after a defined time interval of user inactivity. Time is measured based on server time, accumulated since the user session was connected."},
      {"sfr_id": "FTP_ITC.1.1", "source_pp": "Korean National Protection Profile for Single Sign On V3.0", "text": "The TSF shall provide a communication channel between itself and another trusted IT product that is logically distinct from other communication channels and provides assured identification of its end points and protection of the channel data from modification or disclosure."},
      {"sfr_id": "FIA_AFL.1.1", "source_pp": "Korean National Protection Profile for Single Sign On V3.0", "text": "The TSF shall detect when a defined (or admin-configurable) number of unsuccessful authentication attempts occur related to a defined list of authentication events."}
    ],
    "requirements": [
      {"req_id": "REQ-001", "statement": "The system shall support password authentication plus at least one additional authentication mechanism, and shall authenticate a user's claimed identity according to defined rules governing how the multiple mechanisms interoperate.", "category": "Security - Authentication", "source_ref": "FIA_UAU.5.1 (Korean National PP for SSO V3.0)", "reasoning": "An SSO system that relies on a single, static authentication factor becomes a single point of failure - if that factor is compromised (e.g. a leaked password), an attacker gains access to every connected service at once, which is precisely the amplified risk SSO introduces. FIA_UAU.5.1 closes this gap by requiring support for multiple authentication mechanisms with clearly defined interaction rules, enabling stronger authentication (e.g. MFA) specifically because SSO concentrates so much access behind one login event."},
      {"req_id": "REQ-002", "statement": "The system shall perform mutual authentication between distinct internal components of the SSO system using a defined, standards-compliant authentication protocol.", "category": "Security - Authentication", "source_ref": "FIA_IMA.1.1 (Korean National PP for SSO V3.0)", "reasoning": "REQ-001 secures the user-facing login, but an SSO system is internally composed of multiple trust-sensitive components (e.g. identity provider and relying service) that communicate with each other - if those components don't mutually authenticate, a rogue or spoofed component could impersonate a legitimate part of the SSO infrastructure itself. FIA_IMA.1.1 closes this internal trust gap."},
      {"req_id": "REQ-003", "statement": "The system shall terminate an interactive session after a defined period of user inactivity, measured using server-side time.", "category": "Security - Session Management", "source_ref": "FTA_SSL.3.1 (Korean National PP for SSO V3.0)", "reasoning": "Because SSO grants access to multiple connected services through one session, an unattended, still-active session is a disproportionately high-value target compared to a single-service login - anyone gaining physical or network access to that session inherits access everywhere. FTA_SSL.3.1 closes this gap by mandating automatic termination after inactivity, and specifically requires server-side (not client-side) time tracking to prevent a compromised client from manipulating the timeout."},
      {"req_id": "REQ-004", "statement": "The system shall provide a logically distinct, identity-assured communication channel between itself and other trusted IT products, protecting channel data from modification or disclosure.", "category": "Security - Communication", "source_ref": "FTP_ITC.1.1 (Korean National PP for SSO V3.0)", "reasoning": "SSO inherently involves passing authentication assertions between the identity provider and relying services - if that channel isn't distinctly protected, an attacker positioned on the network could intercept or forge assertions, effectively bypassing REQ-001/002 entirely without ever attacking the authentication logic itself. FTP_ITC.1.1 closes this gap by requiring a dedicated, tamper- and disclosure-protected channel for this inter-component trust traffic."},
      {"req_id": "REQ-005", "statement": "The system shall detect when a defined number of unsuccessful authentication attempts occur for a given authentication event.", "category": "Security - Authentication", "source_ref": "FIA_AFL.1.1 (Korean National PP for SSO V3.0)", "reasoning": "Without this requirement, an attacker could attempt unlimited password guesses against the single SSO login point (REQ-001) with no detection - and because SSO concentrates access to many services behind that one login, a successful brute-force here is far more damaging than against any single service. FIA_AFL.1.1 closes this gap by requiring detection of repeated failed attempts, enabling a response (e.g. lockout) before an attacker can exhaust the credential space."}
    ]
  }
]

with open("agent1_training_dataset_final.json", "w") as f:
    json.dump(dataset_content, f)

# 3. Load Model & Tokenizer in 4-bit
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = 2048,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 4. Dataset Formatting
def format_qwen(example):
    user_prompt = f"Functionality: {example['functionality']}\n\nRetrieved Context:\n"
    for ctx in example['retrieved_context']:
        user_prompt += f"- {ctx['sfr_id']} ({ctx['source_pp']}): {ctx['text']}\n"
    assistant_response = json.dumps(example['requirements'], indent=2)
    messages = [
        {"role": "system", "content": "You are a requirements engineering assistant for IT Security. Given a functionality and Common Criteria context, output a JSON array of requirements with specific security reasoning."},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": assistant_response}
    ]
    return {"messages": messages}

raw_dataset = Dataset.from_json("agent1_training_dataset_final.json")
chat_dataset = raw_dataset.map(format_qwen)

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def apply_template(examples):
    texts = [tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False) for msg in examples["messages"]]
    return {"text": texts}

train_dataset = chat_dataset.map(apply_template, batched=True)

# 5. Execute SFTTrainer with the stable API
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = 2048,
        dataset_num_proc = 2,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

## Save Adapter & Run Generalization Tests (Audit Logging & Firmware Update)

In [ ]:
# 1. Save the fine-tuned LoRA adapters locally
model.save_pretrained("qwen2.5-3b-cc-agent1")
tokenizer.save_pretrained("qwen2.5-3b-cc-agent1")

# 2. Enable native 2x faster inference
FastLanguageModel.for_inference(model)

# 3. Format a test prompt using a totally new domain requirement (Secure Audit Logging)
test_messages = [
    {"role": "system", "content": "You are a requirements engineering assistant for IT Security. Given a functionality and Common Criteria context, output a JSON array of requirements with specific security reasoning."},
    {"role": "user", "content": "Functionality: Secure Audit Logging\n\nRetrieved Context:\n- FAU_GEN.1.1 (General Audit): The TSF shall be able to generate an audit record of the following auditable events: Start-up and shutdown of the audit functions."}
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

# 4. Generate the response
outputs = model.generate(input_ids = inputs, max_new_tokens = 1024, use_cache = True)
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

print(response)

In [ ]:
# Test Case: Secure Firmware Update
test_messages = [
    {"role": "system", "content": "You are a requirements engineering assistant for IT Security. Given a functionality and Common Criteria context, output a JSON array of requirements with specific security reasoning."},
    {"role": "user", "content": """Functionality: Secure Firmware Update

Retrieved Context:
- FPT_TUD_EXT.1.1 (Trusted Update): The TSF shall provide authorized administrators the ability to query the current version of the TOE firmware/software.
- FPT_TUD_EXT.1.2 (Trusted Update): The TSF shall provide authorized administrators the ability to initiate updates to TOE firmware/software.
- FPT_TUD_EXT.1.3 (Trusted Update): The TSF shall provide a means to verify firmware/software updates to the TOE using a digital signature mechanism prior to installing those updates."""}
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 1024, use_cache = True)
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

print(response)

In [ ]:
!zip -r qwen2.5-3b-cc-agent1.zip qwen2.5-3b-cc-agent1/

## Optional v2 Extension: Grounded + Negative Abstention Training

Trains on `agent1_training_dataset_v2.json` with empty `[]` negative examples so the model refuses to hallucinate requirements when no relevant source material is retrieved.

In [ ]:
import json
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

with open("agent1_training_dataset_v2.json", "r") as f:
    full_dataset = json.load(f)

positive_pairs = full_dataset["positive_examples"]
negative_examples = full_dataset["negative_examples"]
print(f"Loaded {len(positive_pairs)} positive pairs, {len(negative_examples)} negative examples")

def format_positive(pair):
    context_text = "\n".join(
        f"- {c['sfr_id']} ({c['source_pp']}): {c['text']}"
        for c in pair["retrieved_context"]
    )
    user_prompt = f"""You are a requirements engineering assistant. Given a functionality and relevant security documentation, generate specific software requirements with reasoning.

Functionality: {pair['functionality']}

Relevant source material:
{context_text}

Generate the security functional requirements with reasoning, citing the specific source SFR for each."""

    reqs_text = json.dumps(pair["requirements"], indent=2)
    assistant_response = f"Based on the retrieved source material, here are the grounded requirements:\n\n{reqs_text}"
    return user_prompt, assistant_response

def format_negative(neg):
    if neg["retrieved_context"]:
        context_text = "\n".join(
            f"- {c['sfr_id']} ({c['source_pp']}): {c['text']}"
            for c in neg["retrieved_context"]
        )
    else:
        context_text = "(no relevant source material was retrieved)"

    user_prompt = f"""You are a requirements engineering assistant. Given a functionality and relevant security documentation, generate specific software requirements with reasoning.

Functionality: {neg['functionality']}

Relevant source material:
{context_text}

Generate the security functional requirements with reasoning, citing the specific source SFR for each."""

    reqs_text = json.dumps(neg["requirements"], indent=2)
    assistant_response = (
        f"Based on the retrieved source material, here are the grounded requirements:\n\n"
        f"{reqs_text}\n\nNote: {neg['note']}"
    )
    return user_prompt, assistant_response

formatted = []
for pair in positive_pairs:
    user_prompt, assistant_response = format_positive(pair)
    messages = [{"role": "user", "content": user_prompt}, {"role": "assistant", "content": assistant_response}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    formatted.append({"text": text})

for neg in negative_examples:
    user_prompt, assistant_response = format_negative(neg)
    messages = [{"role": "user", "content": user_prompt}, {"role": "assistant", "content": assistant_response}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    formatted.append({"text": text})

dataset = Dataset.from_list(formatted)
print(f"Total formatted training examples: {len(dataset)} ({len(positive_pairs)} positive + {len(negative_examples)} negative)")
print("\n--- Example NEGATIVE formatted training text (first 1000 chars) ---")
neg_example_idx = len(positive_pairs)
print(dataset[neg_example_idx]["text"][:1000])

sft_config = SFTConfig(
    output_dir="./agent1-lora-checkpoints-v2",
    num_train_epochs=8,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    max_length=2048,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
)

trainer.train()

print("\nTraining complete.")
model.save_pretrained("./agent1-lora-final-v2")
tokenizer.save_pretrained("./agent1-lora-final-v2")
print("Saved LoRA adapter to ./agent1-lora-final-v2")